In [15]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import plotly.graph_objects as go

# shared style
PALETTE = ["#2166ac", "#4393c3", "#92c5de", "#f4a582", "#d6604d", "#b2182b"]
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
})

DATA = "../../assets/data"

scatter = pd.read_json(f"{DATA}/scatter_distance_fare.json")
r2     = json.load(open(f"{DATA}/r2_comparison.json"))
bubble = pd.read_json(f"{DATA}/hhi_bubble.json")
mono   = pd.read_json(f"{DATA}/monopoly_routes.json")

hhi_by_airport = pd.read_csv("../../data/airport_hhi.csv")

Linked View Plot

In [3]:
import altair as alt
import pandas as pd
import numpy as np

# Drop non-serializable list column before passing to Altair/Vega
map_data = us_airports.drop(columns=["carriers"]).copy()
map_data["TOTAL_PAX_SQRT"] = np.sqrt(map_data["TOTAL_PAX"].fillna(0))

# Shared click selection — initialized to _init_ap, bidirectional across both charts
selection = alt.selection_point(
    fields=["AIRPORT"],
    value=[{"AIRPORT": _init_ap}],
    empty=False,
)

# ── Map ───────────────────────────────────────────────────────────────────────
states_url = "https://vega.github.io/vega-datasets/data/us-10m.json"

background = alt.Chart(alt.topo_feature(states_url, "states")).mark_geoshape(
    fill=PBG, stroke="white", strokeWidth=0.6
)

map_dots = alt.Chart(map_data).mark_circle(stroke="white", strokeWidth=0.7).encode(
    longitude="LON:Q",
    latitude="LAT:Q",
    size=alt.Size("TOTAL_PAX_SQRT:Q", scale=alt.Scale(range=[30, 500]), legend=None),
    color=alt.Color(
        "CARRIER_COUNT:Q",
        scale=alt.Scale(range=[BLUE_L, BLUE]),
        legend=alt.Legend(title="# Carriers"),
    ),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.45)),
    tooltip=[
        alt.Tooltip("AIRPORT:N", title="Airport"),
        alt.Tooltip("HHI_FMT:N", title="HHI"),
        alt.Tooltip("FARE_FMT:N", title="Avg. Price"),
        alt.Tooltip("PASSENGERS_LABEL:N", title="Passengers"),
    ],
).add_params(selection)

# Red highlight dot layered on top of selected airport
map_highlight = alt.Chart(map_data).mark_circle(
    color=RED, stroke="white", strokeWidth=2.5
).encode(
    longitude="LON:Q",
    latitude="LAT:Q",
    size=alt.value(280),
    opacity=alt.value(1.0),
).transform_filter(selection)

map_chart = (background + map_dots + map_highlight).project("albersUsa").properties(
    width=500, height=340,
    title=alt.Title("Airport Market Structure Map", anchor="start"),
)

# ── Scatter ───────────────────────────────────────────────────────────────────
fit_df = pd.DataFrame({"HHI": _fit_x, "AVG_FARE_AIRPORT": _fit_y})

fit_line = alt.Chart(fit_df).mark_line(
    color=GRAY, strokeDash=[5, 5], strokeWidth=1.5, opacity=0.7
).encode(x="HHI:Q", y="AVG_FARE_AIRPORT:Q")

_y_range = sc_df["AVG_FARE_AIRPORT"].max() - sc_df["AVG_FARE_AIRPORT"].min()
r2_ann = alt.Chart(pd.DataFrame({
    "HHI": [sc_df["HHI"].max()],
    "AVG_FARE_AIRPORT": [sc_df["AVG_FARE_AIRPORT"].min() + _y_range * 0.05],
    "label": [f"R² = {_r2:.3f}"],
})).mark_text(align="right", baseline="bottom", color="#333", size=11).encode(
    x="HHI:Q", y="AVG_FARE_AIRPORT:Q", text="label:N"
)

scatter_pts = alt.Chart(sc_df).mark_circle(stroke="white", strokeWidth=0.6).encode(
    x=alt.X("HHI:Q", title="HHI (Herfindahl-Hirschman Index)"),
    y=alt.Y("AVG_FARE_AIRPORT:Q", title="Avg. Flight Price ($)", axis=alt.Axis(format="$,.0f")),
    color=alt.condition(selection, alt.value(RED), alt.value(BLUE)),
    size=alt.condition(selection, alt.value(160), alt.value(55)),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.45)),
    tooltip=[
        alt.Tooltip("AIRPORT:N", title="Airport"),
        alt.Tooltip("HHI:Q", title="HHI", format=".3f"),
        alt.Tooltip("AVG_FARE_AIRPORT:Q", title="Avg. Price", format="$,.2f"),
    ],
).add_params(selection)

scatter_chart = (fit_line + scatter_pts + r2_ann).properties(
    width=380, height=340,
    title=alt.Title("Avg. Flight Price vs. HHI", anchor="start"),
)

# ── Combine ───────────────────────────────────────────────────────────────────
chart = (
    alt.hconcat(map_chart, scatter_chart, spacing=30)
    .resolve_scale(color="independent", size="independent")
    .configure_view(fill=PBG, stroke=None)
    .configure_axis(gridColor=GRAY_L, gridWidth=1, labelFontSize=11, titleFontSize=12)
    .configure_title(fontSize=13, fontWeight="bold")
)

chart


/Users/akshayarun/.pyenv/versions/3.10.13/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: UserWarning:

Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891



alt.HConcatChart(...)